# 3 — Espaço, gravidade e segregação

Com a antena como nó, o mapa deixa de ser pano de fundo: **cada célula de Voronoi é um nó** e
cada linha desenhada é uma aresta. Isso abre análises que a rede de usuários não permitia —
modelo de gravidade, resíduos de fluxo, insularidade e balanço emissor/receptor por região.

## Preparação

In [ ]:
import sys
from pathlib import Path

# permite rodar o notebook a partir de notebooks/ usando os módulos de src/
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.utils import load_config
from src.exporter import InlineExporter
from src import antenna

CIDADE = "campinas"   # troque aqui: precisa existir config/<cidade>.yaml

config = load_config(CIDADE)
config["spatial"]["download_basemap"] = True   # False para rodar offline

edges_antenna = pd.read_parquet(ROOT / config["data"]["edges_antenna_path"])
antennas = pd.read_parquet(ROOT / config["data"]["antennas_path"])

net = antenna.build(edges_antenna, antennas, config)
ex = InlineExporter(config)
print(f"{net.n_antennas} regiões | {net.G.number_of_edges()} fluxos | "
      f"{net.nodes['n_users'].sum():,} moradores agregados")

## Macro-regiões (necessárias para um dos mapas)

Rodamos a topologia primeiro só para obter a coluna `macro_region`.

In [ ]:
from src.pipeline import topology
from src.exporter import InlineExporter

silencioso = InlineExporter(config)
silencioso.save_figure = lambda fig, *a, **k: plt.close(fig)   # sem refazer as figuras do NB2
nodes = topology.run(net, config, silencioso)["nodes"]

## Todos os mapas, a gravidade e a homofilia

A célula abaixo roda o módulo espacial completo: Voronoi por quintil, insularidade, balanço
emissor/receptor, chamadas por morador, macro-regiões, corredores, modelo de gravidade,
resíduos e homofilia socioeconômica.

In [ ]:
from src.pipeline import spatial

resultado = spatial.run(net, config, ex, nodes=nodes)

## O modelo de gravidade

O fluxo entre duas regiões cresce com o tamanho delas e cai com a distância:

$$F_{ij} \approx C \cdot (n_i n_j)^a / d_{ij}^{\,b}$$

O expoente `b` formaliza o "decaimento com a distância" — em vez de só mostrar a curva
caindo, diz **quanto** ela cai. O que sobra do ajuste (o resíduo) é o mais interessante:
pares de bairros que falam muito mais do que tamanho e distância explicariam.

In [ ]:
ex.data["gravity_top_residuals.csv"].head(15)

## Homofilia socioeconômica: cuidado com a leitura

No nível das regiões, a homofilia por quintil é **muito menor** do que parecia no nível das
pessoas. O motivo está no modelo nulo, e o notebook 4 desmonta isso em detalhe.

O que sobrevive, e é mais interessante, é a **assimetria entre os estratos**: o índice de
auto-preferência mostra as regiões ricas se fechando e as pobres se dispersando pelos demais
estratos.

In [ ]:
pd.Series(resultado["metrics"].get("self_preference_index", {})).rename(
    "volume interno observado / esperado"
).to_frame()